in this file i will predict whether a person survived or not without using pipelines and column transformer

Import NumPy for numerical arrays and pandas for loading and working with tabular data.

In [1]:
import numpy as np
import pandas as pd


Import the scikit-learn tools needed for splitting data, filling missing values, encoding categories, scaling, and training a decision tree.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import minmax_scale
from sklearn.tree import DecisionTreeClassifier


Load the Titanic training dataset into a DataFrame.
`sample(5)` previews five random rows so you can inspect the raw data.

In [3]:
df = pd.read_csv("../../datasets/train.csv")
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
70,71,0,2,"Jenkin, Mr. Stephen Curnow",male,32.0,0,0,C.A. 33111,10.500,NaN,S
317,318,0,2,"Moraweck, Dr. Ernest",male,54.0,0,0,29011,14.000,NaN,S
259,260,1,2,"Parrish, Mrs. (Lutie Davis)",female,50.0,0,1,230433,26.000,NaN,S
664,665,1,3,"Lindqvist, Mr. Eino William",male,20.0,1,0,STON/O 2. 3101285,7.925,NaN,S
884,885,0,3,"Sutehall, Mr. Henry Jr",male,25.0,0,0,SOTON/OQ 392076,7.050,NaN,S


Show all column names in the dataset.
This helps decide which columns are useful features and which should be removed.

In [4]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

Drop identifier/text-heavy columns that are not being used for this simple model.
`head(5)` previews the remaining columns after cleanup.

In [5]:
df.drop(columns = ['PassengerId',  "Name", "Ticket" , "Cabin"] , inplace = True)
df.head(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


Count missing values in each column.
This tells you which features need imputation before training the model.

#it shows the total number of vacant columns

In [6]:
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

Split the data into input features `X` and target labels `Y`.
The test set is kept aside to evaluate the model on unseen data.

we have to split the data to train the model

In [7]:
X_train , X_test , Y_train , Y_test = train_test_split(df.drop(columns = ["Survived"]), df["Survived"] , test_size= 0.2, random_state = 42)

Preview three random rows from the training features.
This is a quick check that the feature split looks correct.

In [8]:
X_train.sample(3)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
707,1,male,42.0,0,0,26.2875,S
694,1,male,60.0,0,0,26.5500,S
774,2,female,54.0,1,3,23.0000,S


Preview three random labels from the training target.
This confirms the target series contains the expected survival values.

In [9]:
Y_train.sample(3)

384    0
627    1
366    1
Name: Survived, dtype: int64

Create imputers for missing values.
Age uses the default mean strategy, while Embarked uses the most frequent category.

In [10]:
si_age = SimpleImputer()
si_embarked = SimpleImputer(strategy= "most_frequent")

Fit the imputers on the training data and transform the missing values.
This produces cleaned Age and Embarked arrays for training.

In [11]:
X_train_age = si_age.fit_transform(X_train[["Age"]])
X_train_embarked = si_embarked.fit_transform(X_train[["Embarked"]])

Fill missing values in the test Age and Embarked columns.
For a strict train-test workflow, this should use `transform` instead of `fit_transform`.

In [12]:
X_test_age = si_age.transform(X_test[["Age"]])
X_test_embarked = si_embarked.fit_transform(X_test[["Embarked"]])

This note identifies Sex and Embarked as nominal categorical columns.
They need one-hot encoding because their values do not have a natural order.

In [13]:
#now i have to apply one hot encoding to sex and embarked column because they are 
#categorical and there is no order between them

Create one-hot encoders for Sex and Embarked.
`handle_unknown='ignore'` prevents errors if test data contains an unseen category.

In [14]:
ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

Fit the encoders on training categories and transform the training categorical columns.
The result is numeric arrays that the model can use.

In [15]:
X_train_sex = ohe_sex.fit_transform(X_train[["Sex"]])
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

Transform the test categorical columns using the already-fitted encoders.
This keeps the train and test feature layouts consistent.

In [16]:
X_test_sex = ohe_sex.transform(X_test[["Sex"]])
X_test_embarked = ohe_embarked.transform(X_test_embarked)

Display the one-hot encoded Sex training array.
This helps verify that male/female values were converted into numeric columns.

In [17]:
X_train_sex

array([[0., 1.],
       [0., 1.],
       [0., 1.],
       ...,
       [0., 1.],
       [1., 0.],
       [0., 1.]], shape=(712, 2))

Display the one-hot encoded Embarked training array.
This helps verify that port categories were converted into numeric columns.

In [18]:
X_train_embarked

array([[0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       ...,
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.]], shape=(712, 3))

Keep the remaining columns that do not need imputation or one-hot encoding here.
The original Age, Sex, and Embarked columns are dropped because transformed versions already exist.

In [19]:
X_train_rem = X_train.drop(columns = ["Age", "Sex" , "Embarked"])
X_test_rem = X_test.drop(columns = ["Age", "Sex" , "Embarked"])

Concatenate all transformed feature blocks into final train and test arrays.
Each row now contains only numeric features ready for the classifier.

In [20]:
X_train_transformed = np.concatenate((X_train_sex, X_train_age, X_train_rem, X_train_embarked) , axis = 1)
X_test_transformed = np.concatenate((X_test_sex, X_test_age, X_test_rem, X_test_embarked) , axis = 1)

Check the shape of the transformed test feature matrix.
This confirms how many rows and final feature columns are available for prediction.

In [21]:
X_test_transformed.shape

(179, 10)

Create a decision tree classifier and train it on the transformed training data.
The model learns patterns that relate passenger features to survival labels.

In [22]:
clf = DecisionTreeClassifier()#creating an object of decision tree classifier
clf.fit(X_train_transformed, Y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,None
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


Use the trained classifier to predict survival for the transformed test data.
Displaying `Y_pred` lets you inspect the predicted labels.

In [23]:
Y_pred = clf.predict(X_test_transformed)
Y_pred

array([0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0,
       0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0,
       0, 1, 1])

Import accuracy scoring and calculate the percentage of correct predictions.
This gives a simple evaluation score for the test set.

In [24]:
from sklearn.metrics import accuracy_score
accuracy_score(Y_test,Y_pred)*100

78.2122905027933

now we have to predict the score